In [1]:
!pip uninstall -qqy jupyterlab
!pip install -U -q "google-genai==1.7.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 6.1 MB/s eta 0:00:00


In [2]:
from google import genai
from google.genai import types


In [3]:
genai.__version__


'1.7.0'

In [4]:
from google.colab import userdata


In [5]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")


In [6]:
from google.api_core import retry

is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in{429, 503})


In [7]:
# This code snippet checks if the genai.models.Models.generate_content function has already been wrapped.
# If it hasn't, it wraps the function using a retry.Retry object configured with a specific is_retriable predicate.
# This effectively adds automatic retrying capabilities to the generate_content function, making calls to the generative AI model more resilient to transient errors like rate limits or temporary server issues.
# The is_retriable function is key, as it defines which specific errors should trigger a retry attempt.

if not hasattr(genai.models.Models.generate_content, "__wrapped__"):
  genai.models.Models.generate_content = retry.Retry(predicate = is_retriable)(genai.models.Models.generate_content)


## Creating a local database


In [8]:
# This is a built-in IPython/Jupyter magic command used to load an "extension".
%load_ext sql
# This command tells the %sql magic where to send subsequent SQL queries.
# It establishes a connection to the SQLite database file named sample.db located in the current working directory of the notebook
%sql sqlite:///sample.db


In [9]:
%%sql
-- Create the 'products' table
CREATE TABLE IF NOT EXISTS products (
  	product_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	product_name VARCHAR(255) NOT NULL,
  	price DECIMAL(10, 2) NOT NULL
  );

-- Create the 'staff' table
CREATE TABLE IF NOT EXISTS staff (
  	staff_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	first_name VARCHAR(255) NOT NULL,
  	last_name VARCHAR(255) NOT NULL
  );

-- Create the 'orders' table
CREATE TABLE IF NOT EXISTS orders (
  	order_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	customer_name VARCHAR(255) NOT NULL,
  	staff_id INTEGER NOT NULL,
  	product_id INTEGER NOT NULL,
  	FOREIGN KEY (staff_id) REFERENCES staff (staff_id),
  	FOREIGN KEY (product_id) REFERENCES products (product_id)
  );


 * sqlite:///sample.db
Done.
Done.
Done.


[]

## Defining database functions

In [10]:
import sqlite3

db_file = "sample.db"
db_connection = sqlite3.connect(db_file)


In [11]:
def list_tables() -> list[str]:
  # Retrieving the names of all the tables in the database
  print(" - DB CALL: list_tables()")
  cursor = db_connection.cursor()

  # Fetching the table names
  cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

  tables = cursor.fetchall()
  return [t[0] for t in tables]


In [12]:
list_tables()


 - DB CALL: list_tables()


['products', 'sqlite_sequence', 'staff', 'orders']

In [13]:
def delete_table_rows():
  print("WARNING: About to delete all rows from all user tables.")

  try:
    table_names = list_tables()
    if not table_names:
      print("No tables found to clear.")
      return

    print(f"Found tables to clear: {', '.join(table_names)}")
    cursor = db_connection.cursor()

    for table_name in table_names:
      try:
        sql = f"DELETE FROM {table_name};"
        print(f"--Executing {sql}--")
        cursor.execute(sql)
        print(f"--Cleared table {table_name}--")
      except Exception as e:
        print(f"--Error clearing table {table_name}: {e}--")

    db_connection.commit()
    print("\nAll rows deleted successfully from specified tables and changes committed.")

  except Exception as e:
    print(f"\nAn error occurred during the deletion process: {e}")

    try:
      db_connection.rollback()
      print("--Attempting to rollback changes--")
    except Exception as e:
      print(f"--Error rolling back: {e}--")


In [14]:
delete_table_rows()


 - DB CALL: list_tables()
Found tables to clear: products, sqlite_sequence, staff, orders
--Executing DELETE FROM products;--
--Cleared table products--
--Executing DELETE FROM sqlite_sequence;--
--Cleared table sqlite_sequence--
--Executing DELETE FROM staff;--
--Cleared table staff--
--Executing DELETE FROM orders;--
--Cleared table orders--

All rows deleted successfully from specified tables and changes committed.


In [15]:
%%sql
-- Insert data into the 'products' table
INSERT INTO products (product_name, price) VALUES
  	('Laptop', 799.99),
  	('Keyboard', 129.99),
  	('Mouse', 29.99);

-- Insert data into the 'staff' table
INSERT INTO staff (first_name, last_name) VALUES
  	('Alice', 'Smith'),
  	('Bob', 'Johnson'),
  	('Charlie', 'Williams');

-- Insert data into the 'orders' table
INSERT INTO orders (customer_name, staff_id, product_id) VALUES
  	('David Lee', 1, 1),
  	('Emily Chen', 2, 2),
  	('Frank Brown', 1, 3);


 * sqlite:///sample.db
3 rows affected.
3 rows affected.
3 rows affected.


[]

In [16]:
# -> list[tuple[str, str]]: This is a type hint indicating the function is expected to return a list (list).
# Each element within that list should be a tuple (tuple) containing exactly two strings ([str, str]).
def describe_table(table_name: str) -> list[tuple[str, str]]:
  print(f" - DB CALL: describe_table({table_name})")
  # Cursors are used to execute SQL commands and fetch data from the database. We need a cursor to interact with the database via the connection.
  cursor = db_connection.cursor()
  # PRAGMA table_info(...): This is a SQLite-specific command. It's not standard SQL. It queries the database for metadata about the columns of the specified table.
  cursor.execute(f"PRAGMA table_info({table_name});")
  # The PRAGMA table_info command returns one row for each column in the table.
  # The list of the tuples is stored in the variable, schema.
  schema = cursor.fetchall()
  return [(col[1], col[2]) for col in schema]


In [17]:
describe_table("products")


 - DB CALL: describe_table(products)


[('product_id', 'INTEGER'),
 ('product_name', 'VARCHAR(255)'),
 ('price', 'DECIMAL(10, 2)')]

In [18]:
def execute_query(sql: str) -> list[list[str]]:
  print(f" - DB CALL: execute_query({sql})")

  cursor = db_connection.cursor()
  cursor.execute(sql)
  return cursor.fetchall()


In [19]:
execute_query("SELECT * FROM products")


 - DB CALL: execute_query(SELECT * FROM products)


[(1, 'Laptop', 799.99), (2, 'Keyboard', 129.99), (3, 'Mouse', 29.99)]

## Implement function calls

In [20]:
# Python functions defined above
db_tools = [list_tables, describe_table, execute_query]


In [21]:
instruction = """You are a helpful chatbot that can interact with an SQL database
for a computer store. You will take the users questions and turn them into SQL
queries using the tools available. Once you have the information you need, you will
answer the user's question using the data returned.

Use list_tables to see what tables are present, describe_table to understand the
schema, and execute_query to issue an SQL SELECT query."""


In [22]:
client = genai.Client(api_key = GOOGLE_API_KEY)


In [23]:
# Starting a chat with automatic function calling enabled
chat = client.chats.create(
    model = "gemini-2.0-flash",
    config = types.GenerateContentConfig(
        system_instruction = instruction,
        tools = db_tools,
    ),
)


In [24]:
response = chat.send_message(
    "What is the cheapest product?"
)
print(f"\n{response.text}")


 - DB CALL: execute_query(SELECT * FROM Products ORDER BY Price ASC LIMIT 1)

The cheapest product is the Mouse, which costs $29.99.


In [26]:
# Engaging in a chat conversation where we can ask about the contents of the database
response = chat.send_message(
    "What products should salesperson Alice focus on to round out her portfolio? Explain why."
)
print(f"\n{response.text}")


 - DB CALL: list_tables()
 - DB CALL: describe_table(staff)
 - DB CALL: describe_table(products)
 - DB CALL: describe_table(orders)
 - DB CALL: execute_query(SELECT staff_id FROM staff WHERE first_name = 'Alice')
 - DB CALL: execute_query(SELECT DISTINCT product_id FROM orders WHERE staff_id = 1)
 - DB CALL: execute_query(SELECT product_name FROM products WHERE product_id NOT IN (SELECT DISTINCT product_id FROM orders WHERE staff_id = 1))

Alice should focus on selling Keyboards. She has sold products with product\_id 1 and 3, and the only other product is a Keyboard. By focusing on Keyboards, she can round out her portfolio and increase her overall sales.



In [27]:
# Engaging in a chat conversation where we can ask about the contents of the database
response = chat.send_message(
    "What products should salesperson Alice focus on to round out her portfolio? Explain why."
)
print(f"\n{response.text}")


 - DB CALL: execute_query(SELECT product_id, product_name FROM products)

Alice has sold Laptops (product ID 1) and Mice (product ID 3). Therefore, to round out her portfolio, Alice should focus on selling Keyboards (product ID 2). This would give her a more diverse sales record across all product types.



## Inspecting the conversation
In order to see the calls that the model makes, and what the client returns in response, we can inspect the chat history. This helper function will print out each turn along with the relevant fields passed or returned.

In [28]:
# Importing a helper library for text formatting (especially for indentation)
import textwrap


In [29]:
def print_chat_turns(chat):
  # Printing each turn in the chat history, including function calls and function responses
  # Get history from the `chat` object
  # chat.get_history() returns a list of 'events' or 'turns' in the conversation
  for event in chat.get_history():
    # Printing who sent this event ('User' or 'Model')
    # event.role will be something like 'user' or 'model'
    # .capitalize() makes the first letter uppercase (e.g., 'User')
    print(f"{event.role.capitalize()}: ")

    # An event might have multiple 'parts' (e.g., text AND a function call)
    # Looping through each part of the current event
    for part in event.parts:
      # Checking to see if this part is plain text
      # The 'walrus operator' (:=) assigns part.text to 'txt' AND checks if 'txt' is not empty/None
      if txt := part.text:
        # If it's text, printing it indented and in quotes
        print(f' "{txt}"')

      # Else, checking to see if this part is a function call requested by the Model
      elif fn := part.function_call:
        # If it's a function call:
        # a. Getting the arguments,
        # b. Formatting them nicely like "key=val, key2=val2"
        args = ", ".join(f"{key} = {val}" for key, val in fn.args.items())
        # c. Print the function name and its formatted arguments
        print(f" Function call: {fn.name}({args})")

      # Else, checking to see if this part is a function response sent back by the code
      elif response := part.function_response:
        # If it's a function response:
        print(f" Function response:")
        # a. Access the actual result data inside the response object
        # (Assuming the result is stored under a 'result' key)
        if 'result' in response.response:
          response_data_string = str(response.response['result'])
          # b. Using textwrap.indent to add extra indentation (4 spaces) to every line of the result
          # This makes multi-line results (like lists) look neat
          indented_response = textwrap.indent(response_data_string, "    ")
          # c. Printing the indented result
          print(indented_response)
        else:
          print("    No 'result' found in the response.")

    # Printing a blank line after each event for better separation
    print()


In [30]:
print_chat_turns(chat)


User: 
 "What is the cheapest product?"

Model: 
 Function call: execute_query(sql = SELECT * FROM Products ORDER BY Price ASC LIMIT 1)

User: 
 Function response:
    [(3, 'Mouse', 29.99)]

Model: 
 "The cheapest product is the Mouse, which costs $29.99."

User: 
 "What products should salesperson Alice focus on to round out her portfolio? Explain why."

Model: 
 Function call: execute_query(sql = SELECT DISTINCT product_name FROM Sales)
 Function call: execute_query(sql = SELECT name FROM Products)

User: 
 Function response:
    No 'result' found in the response.
 Function response:
    No 'result' found in the response.

Model: 
 "I am sorry, I cannot answer this question. The database does not have the required information. There is no Sales table, and the Products table does not have a column called 'name'."

User: 
 "What products should salesperson Alice focus on to round out her portfolio? Explain why."

Model: 
 Function call: list_tables()

User: 
 Function response:
    ['p

## Compositional function calling
A powerful new feature in Gemini 2.0 is the model's ability to compose user-provided function calls together while generating code. This means that the model is able to take the available tools, generate code that uses it, and execute it all. The feature requires the Live API. As the Multimodal Live API is a bi-directional streaming service, everything is set up in advance and then executed. This is a little more complex but the result is quite powerful. First, we will define a function that will handle streaming model output. It will stream text output, handle tool-calling and show the generated code that the model writes and executes to fulfill the task.

In [31]:
# Importing libraries for pretty printing and displaying rich output in environments like Jupyter
from pprint import pformat
from IPython.display import display, Image, Markdown


In [34]:
# Defining an asynchronous function. This function can pause and wait for I/O without blocking
async def handle_response(stream, tool_impl = None):
  # Streaming output and handling any tool calls during the session
  # Keeping track of all message chunks received in this interaction
  all_responses = []

  # Looping through the messages as they arrive from the stream
  # 'await stream.receive()' waits for the next message chunk
  async for msg in stream.receive():
    # Store the raw message chunk
    all_responses.append(msg)

    # -- Checking to see what type of content is in the message chunk

    # If the chunk contains text
    if text := msg.text:
      # Outputting any text chunks that are streamed
      # A small check to only display the ' ### Text' header once per block of text
      if len(all_responses) < 2 or not all_responses[-2].text:
        # Displaying a header if this is the first message chunk
        display(Markdown(' ### Text'))

      # Printing the text chunk. 'end = ""' prevents adding extra newlines, allowing streamed text to appear smoothly on one line.
      print(text, end = "")

    # If the chunk contains a tool call request from the model,
    elif tool_call := msg.tool_call:
      # Handling tool call requests
      # A single message might request multiple function calls,
      for fc in tool_call.function_calls:
        # Displaying the header
        display(Markdown(' ### Tool call'))

        # Executing the tool and collecting the result to return to the model
        # Checking to see if we were given an actual function/object to run tools
        if callable(tool_impl):
          try:
            # This is where the actual function gets called
            # `fc.args` contains the arguments provided by the model
            # `**fc.args` unpacks this dictionary into keyword arguments for the function
            result = tool_impl(**fc.args)
          except Exception as e:
            # If the function fails, capture the error message as the result
            result = str(e)

        # If no tool implementation was provided, just send back 'ok'
        else:
          result = "OK!"


        # Sending the result back to the model
        # Constructing the specific response object the API expects

        tool_response = types.LiveClientToolResponse(
            function_responses = [types.FunctionResponse(
                # The name of the function that was called
                name = fc.name,
                # The ID from the model's request, to match request/response. This is very important
                id = fc.id,
                # Packaging the actual result
                response = {'result' : result},
            )]
        )
        # Sending the result back through the same stream
        await stream.send(input = tool_response)

     # If the chunk contains content related to server-side code execution
    elif msg.server_content and msg.server_content.model_turn:

      # Printing any messages showing code the model generated and ran
      # This section handles cases where the model itself might generate/execute code
      for part in msg.server_content.model_turn.parts:
        if code := part.executable_code:
          # Display code the model generated
          display(Markdown(
              f" ### Code \n ``` \n {code.code} \n```"
          ))

        elif result := part.code_execution_result:
          # Displaying the result of code the model executed
          display(Markdown(f"### Result: {result.outcome}\n"
                           f"```\n{pformat(result.output)}\n```"))

        elif img := part.inline_data:
          # Displaying any images generated by the model's code execution
          display(Image(img.data))

  # After the stream ends, printing a final newline for cleaner output
  print()
  # Returning the list of all raw message chunks received
  return all_responses


In simple terms, imagine you're having a live phone call (stream) with the AI. This function acts as your real-time assistant listening to the call.


*   Listen: It waits (await stream.receive()) for the AI to say something.
*   Record: It writes down everything the AI says (all_responses.append(msg)).
*   Handle Speech: If the AI just talks (msg.text), your assistant repeats it out loud (print(text)).
*   Handle Requests: If the AI asks you to do something specific (a tool_call, like "look up the weather for Tokyo"), your assistant:
  - Identifies the request (fc.name, fc.args).
  - Uses the provided tool (tool_impl) to actually do the task (e.g., calls your weather lookup function).
  - Notes down the result (or any error).
  - Tells the AI the result (await stream.send(...)), making sure to reference the AI's original request ID (fc.id) so the AI knows which request the result belongs to.
*   Handle AI's Own Actions: (Less common for user-defined tools) If the AI mentions it ran some code on its end, your assistant displays that code and its result.
*   End Call: Once the AI finishes talking (stream ends), the assistant adds a final newline and gives you the complete transcript (return all_responses).

This function is essential for interactive function calling where the AI might ask your code to perform actions mid-conversation. It manages the back-and-forth required for tool use within a streaming context.

## Textual live database chat

In [37]:
model = "gemini-2.0-flash"
live_client = genai.Client(api_key = GOOGLE_API_KEY,
                           http_options = types.HttpOptions(apiVersion = "v1alpha"))

# Wrapping the existing execute_query tool
execute_query_tool_def = types.FunctionDeclaration.from_callable(
    client = live_client,
    callable = execute_query,
)

# Providing the model with enough information to use the tool, such as describing the database so it understands which SQL syntax to use.
system_intent = """You are a database interface. Use the `execute_query` function
to answer the users questions by looking up information in the database,
running any necessary queries and responding to the user.

You need to look up table schema using sqlite3 syntax SQL, then once an
answer is found be sure to tell the user. If the user is requesting an
action, you must also execute the actions.
"""

config = {
    "response_modalities": ["TEXT"],
    "system_instruction": {"parts": [{"text": system_intent}]},
    "tools": [
        {"code_execution": {}},
        {"function_declarations": [execute_query_tool_def.to_json_dict()]},
    ],
}


In [38]:
async with live_client.aio.live.connect(model = model, config = config) as session:

  message = "Please generate and insert 5 new rows in the orders table."
  print(f"> {message}\n")

  await session.send(input = message, end_of_turn = True)
  await handle_response(session, tool_impl = execute_query)


ConnectionClosedError: received 1008 (policy violation) models/gemini-2.0-flash is not found for API version v1alpha, or is not supported for bidiGenerateContent. Call ListModels ; then sent 1008 (policy violation) models/gemini-2.0-flash is not found for API version v1alpha, or is not supported for bidiGenerateContent. Call ListModels 

In [39]:
async with live_client.aio.live.connect(model = model, config = config) as session:

  message = "Can you figure out the number of orders that were made by each of the staff?"
  print(f"> {message}\n")

  await session.send(input = message, end_of_turn = True)
  await handle_response(session, tool_impl = execute_query)

  message = "Generate and run some code to plot this as a python seaborn chart."
  print(f"> {message}\n")

  await session.send(input = message, end_of_turn = True)
  await handle_response(session, tool_impl = execute_query)


ConnectionClosedError: received 1008 (policy violation) models/gemini-2.0-flash is not found for API version v1alpha, or is not supported for bidiGenerateContent. Call ListModels ; then sent 1008 (policy violation) models/gemini-2.0-flash is not found for API version v1alpha, or is not supported for bidiGenerateContent. Call ListModels 